In [9]:
import pandas as pd
import numpy as np

customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
transactions = pd.read_csv("transactions.csv")

for name, df in {"Customers": customers, "Products": products, "Transactions": transactions}.items():
    print(f"\n=== {name} ===")
    display(df.head())
    print(df.info())




=== Customers ===


,customer_id,name,email,registration_date,country,age
0,C001,Logan Brown,NaN,2024-01-01,Canada,39
1,C002,John Rodriguez,emma.johnson1@email.com,2024-01-02,France,28
2,C003,Ava Davis,NaN,2024-01-04,Australia,65
3,C004,William Brown,NaN,2024-01-06,Italy,33
4,C005,Abigail Moore,william.jones4@email.com,2024-01-08,Canada,50


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        205 non-null    object
 1   name               205 non-null    object
 2   email              185 non-null    object
 3   registration_date  205 non-null    object
 4   country            205 non-null    object
 5   age                205 non-null    object
dtypes: object(6)
memory usage: 9.7+ KB
None

=== Products ===


,product_id,product_name,category,price,stock
0,P001,Speaker,Electronics,353.96,15
1,P002,Science Book,Books,34.88,11246
2,P003,Sweater,Clothing,23.53,97
3,P004,Smartphone,Electronics,56.05,86
4,P005,Running Shoes,sports,-339.29,50


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    50 non-null     object 
 1   product_name  50 non-null     object 
 2   category      50 non-null     object 
 3   price         47 non-null     float64
 4   stock         50 non-null     int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 2.1+ KB
None

=== Transactions ===


,transaction_id,customer_id,product_id,quantity,transaction_date,payment_method
0,T001,C178,P002,1.0,2024-01-01,Credit Card
1,T002,C163,P015,5.0,2024-01-01,PayPal
2,T003,C124,P011,1.0,2024-01-02,Credit Card
3,T004,C033,P008,2.0,2024-01-03,Bank Transfer
4,T005,C161,P026,1.0,2024-01-03,Credit Card


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    508 non-null    object 
 1   customer_id       508 non-null    object 
 2   product_id        508 non-null    object 
 3   quantity          492 non-null    float64
 4   transaction_date  508 non-null    object 
 5   payment_method    508 non-null    object 
dtypes: float64(1), object(5)
memory usage: 23.9+ KB
None


In [10]:
summary = []
for name, df in {"Customers": customers, "Products": products, "Transactions": transactions}.items():
    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isna().sum().sum(),
        "Duplicates": df.duplicated().sum()
    })

pd.DataFrame(summary)


,Dataset,Rows,Columns,Missing Values,Duplicates
0,Customers,205,6,20,4
1,Products,50,5,3,0
2,Transactions,508,6,16,6


In [11]:

print(customers['country'].value_counts())
print(customers['age'].describe())

print(products['category'].value_counts())
print(products.groupby('category')['price'].mean())
print(products[products['stock'] == 0]['product_name'])

print(transactions['payment_method'].value_counts())
top_product = transactions['product_id'].value_counts().idxmax()
print("Most popular product:", top_product)
top_customer = transactions['customer_id'].value_counts().idxmax()
print("Top customer:", top_customer)


country
Canada            27
Australia         23
Germany           21
Italy             20
Japan             20
France            19
Netherlands       19
United Kingdom    17
Spain             15
United States     10
US                 8
USA                6
Name: count, dtype: int64
count     205
unique     70
top        50
freq       10
Name: age, dtype: object
category
Books          13
Electronics    10
Clothing        9
Home            8
books           3
Sports          2
sports          2
home            2
electronics     1
Name: count, dtype: int64
category
Books          147.917500
Clothing       177.265556
Electronics     84.778889
Home           254.864286
Sports         295.710000
books          222.910000
electronics    324.790000
home           202.840000
sports         -58.430000
Name: price, dtype: float64
23           Camera
24    Fiction Novel
27     Fantasy Book
Name: product_name, dtype: object
payment_method
PayPal           173
Credit Card      166
Bank Transfer 

In [13]:
class DataCleaner:
    def __init__(self, customers, products, transactions):
        self.customers = customers.copy()
        self.products = products.copy()
        self.transactions = transactions.copy()

    def clean_customers(self):
        self.customers = self.customers.dropna(subset=['email'])
        self.customers = self.customers.drop_duplicates()
        self.customers['age'] = pd.to_numeric(
            self.customers['age'].astype(str).str.extract(r'(\d+)', expand=False),
            errors='coerce'
        ).astype('Int64')
        mapping = {'usa': 'United States', 'us': 'United States'}
        self.customers['country'] = (self.customers['country']
                                     .astype(str).str.strip().str.lower()
                                     .map(mapping).fillna(self.customers['country'].str.title()))
        self.customers['email'] = self.customers['email'].str.lower()
        return self.customers

    def clean_products(self):
        self.products['price'] = pd.to_numeric(self.products['price'], errors='coerce')
        self.products.loc[self.products['price'] < 0, 'price'] = np.nan
        self.products['price'] = self.products.groupby('category')['price'].transform(
            lambda x: x.fillna(x.median())
        )
        self.products.loc[self.products['stock'] > 500, 'stock'] = 500
        self.products['category'] = self.products['category'].str.title()
        self.products['product_name'] = self.products['product_name'].str.strip()
        self.products = self.products.drop_duplicates()
        return self.products

    def clean_transactions(self):
        mode_val = self.transactions['quantity'].mode()[0]
        self.transactions['quantity'] = self.transactions['quantity'].fillna(mode_val)
        self.transactions = self.transactions.drop_duplicates(subset=['transaction_id'])
        self.transactions = self.transactions[self.transactions['customer_id'].isin(self.customers['customer_id'])]
        self.transactions['transaction_date'] = pd.to_datetime(self.transactions['transaction_date'], errors='coerce')
        self.transactions = self.transactions[self.transactions['transaction_date'] <= '2024-12-31']
        self.transactions['payment_method'] = self.transactions['payment_method'].str.title().str.strip()
        return self.transactions

    def run_all(self):
        self.clean_customers()
        self.clean_products()
        self.clean_transactions()
        return self.customers, self.products, self.transactions

cleaner = DataCleaner(customers, products, transactions)
customers_clean, products_clean, transactions_clean = cleaner.run_all()


In [15]:
for name, df in {"Customers": customers_clean, "Products": products_clean, "Transactions": transactions_clean}.items():
    print(f"\n{name} missing:\n", df.isna().sum())
    print("Duplicates:", df.duplicated().sum())

customers_clean.to_csv("customers_clean.csv", index=False)
products_clean.to_csv("products_clean.csv", index=False)
transactions_clean.to_csv("transactions_clean.csv", index=False)



Customers missing:
 customer_id          0
name                 0
email                0
registration_date    0
country              0
age                  0
dtype: int64
Duplicates: 1

Products missing:
 product_id      0
product_name    0
category        0
price           0
stock           0
dtype: int64
Duplicates: 0

Transactions missing:
 transaction_id      0
customer_id         0
product_id          0
quantity            0
transaction_date    0
payment_method      0
dtype: int64
Duplicates: 0


In [16]:
merged = (transactions_clean
          .merge(customers_clean, on='customer_id', how='inner')
          .merge(products_clean, on='product_id', how='inner'))
merged.head()


,transaction_id,customer_id,product_id,quantity,transaction_date,payment_method,name,email,registration_date,country,age,product_name,category,price,stock
0,T001,C178,P002,1.0,2024-01-01,Credit Card,Michael Wilson,isabella.taylor177@email.com,2024-11-20,United Kingdom,50,Science Book,Books,34.88,500
1,T003,C124,P011,1.0,2024-01-02,Credit Card,Ava Rodriguez,sophia.rodriguez123@email.com,2024-08-13,Canada,50,Smartphone,Electronics,273.17,100
2,T005,C161,P026,1.0,2024-01-03,Credit Card,Amelia Jones,lucas.white160@email.com,2024-10-20,Spain,75,Cookbook,Books,226.75,95
3,T006,C189,P045,4.0,2024-01-04,Bank Transfer,John Anderson,logan.jones188@email.com,2024-12-10,Spain,67,Fantasy Book,Books,256.54,500
4,T007,C179,P044,3.0,2024-01-05,Credit Card,Abigail Jackson,liam.moore178@email.com,2024-11-22,United Kingdom,26,Camera,Electronics,17.50,18


In [17]:
merged['total_amount'] = merged['price'] * merged['quantity']
merged['discount'] = np.where(merged['quantity'] > 3, 0.1 * merged['total_amount'], 0)
merged['final_amount'] = merged['total_amount'] - merged['discount']
merged['transaction_month'] = merged['transaction_date'].dt.month_name()
merged['transaction_day'] = merged['transaction_date'].dt.day_name()
merged['is_weekend'] = merged['transaction_day'].isin(['Saturday', 'Sunday'])
merged['customer_segment'] = pd.cut(
    merged['final_amount'], bins=[0, 500, 1000, np.inf],
    labels=['Low', 'Medium', 'High']
)


In [18]:
print(merged.groupby('category')['final_amount'].sum().sort_values(ascending=False))

print(merged.groupby('product_name')['final_amount'].sum().nlargest(10))

print(merged.groupby('country')['final_amount'].sum().nlargest(5))


category
Books          78372.2930
Home           47883.2600
Clothing       40513.4375
Electronics    40248.2350
Sports         23044.1090
Name: final_amount, dtype: float64
product_name
Cookbook         28505.666
Fiction Novel    24873.314
Speaker          11751.472
Picture Frame    11619.830
Smartphone       11397.265
Plant Pot        11277.968
Blanket          10994.306
Fantasy Book     10324.152
Camera            9885.903
Scarf             9712.881
Name: final_amount, dtype: float64
country
Canada           29540.4215
Italy            28180.7360
Germany          26029.8700
Australia        25983.2140
United States    25324.3725
Name: final_amount, dtype: float64
